# Buzzlytics — Build the datasets (RUN ONCE)

Two-stage design, so this builds **two** zips into your Google Drive:

- **`bee_dataset.zip`** — detection (stage 1): VnPollenBee boxes, classes `bee` + `pollen_bee`.
- **`varroa_cls.zip`** — classification (stage 2): VarroaDataset crops, `healthy` / `varroa`.

Varroa is NOT a detection class — it's decided per-bee by a classifier on each detected bee
crop (the detect-then-classify approach). After this runs once, just paste each zip's share
link into its training notebook.

## Run
1. Runtime → Run all. Approve the two Google popups (API auth + Drive mount).
2. At the end: share-link each zip (right-click → Share → **Anyone with the link** → Copy link).
3. `bee_dataset.zip` → `colab_train.ipynb`; `varroa_cls.zip` → `colab_train_varroa_cls.ipynb`.

In [ ]:
# === 0. Config ===
REPO_URL = "https://github.com/okayxsh/Buzzlytics---CSCI435.git"
REPO_DIR = "/content/Buzzlytics---CSCI435"
VPB_FOLDER_ID = "1fdEcu7CNmEkVAamu9wh_Ppw_-uW3VNY1"   # VnPollenBee Drive folder
VARROA_LIMIT = 3000   # cap redundant varroa crops (class-balanced). None = all 13.5k

In [ ]:
# === 1. Install deps + clone repo (for the converters) ===
!pip -q install gdown pyyaml
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# === 2. Pull VnPollenBee LabelMe JSONs via the Drive API (parallel) ===
import io, shutil, threading
from concurrent.futures import ThreadPoolExecutor
from google.colab import auth
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
auth.authenticate_user()
VPB_DIR = "/content/vnpollenbee"

def _api():
    return gbuild("drive", "v3", cache_discovery=False)

def _children(api, fid):
    items, tok = [], None
    while True:
        r = api.files().list(q=f"'{fid}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)", pageSize=1000,
            pageToken=tok, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        items += r["files"]; tok = r.get("nextPageToken")
        if not tok: break
    return items

def _walk(api, fid, dest):
    os.makedirs(dest, exist_ok=True)
    jobs = []
    for f in _children(api, fid):
        if f["mimeType"] == "application/vnd.google-apps.folder":
            jobs += _walk(api, f["id"], os.path.join(dest, f["name"]))
        elif f["name"].lower().endswith(".json"):
            jobs.append((f["id"], os.path.join(dest, f["name"])))
    return jobs

_tl = threading.local()
def _get(job):
    if not hasattr(_tl, "api"): _tl.api = _api()
    fid, path = job
    dl = MediaIoBaseDownload(io.FileIO(path, "wb"), _tl.api.files().get_media(fileId=fid))
    done = False
    while not done: _, done = dl.next_chunk()

shutil.rmtree(VPB_DIR, ignore_errors=True)
print("Enumerating VnPollenBee folder...")
jobs = _walk(_api(), VPB_FOLDER_ID, VPB_DIR)
print(f"  {len(jobs)} JSONs -> downloading in parallel...")
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_get, jobs))
print("  done")

In [ ]:
# === 3. Build BOTH datasets INLINE (errors show here directly) ===
# Stage 1: detection (VnPollenBee -> bee/pollen boxes)  -> datasets/data
# Stage 2: varroa classification crops (VarroaDataset)  -> datasets/varroa_cls
import sys
from pathlib import Path
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
from datasets.prepare_dataset import (
    prepare_vnpollenbee, prepare_varroa, write_data_yaml, _ensure_dirs)

RAW = Path("datasets/raw"); DET = Path("datasets/data"); CLS = Path("datasets/varroa_cls")
shutil.rmtree(DET, ignore_errors=True); shutil.rmtree(CLS, ignore_errors=True)
RAW.mkdir(parents=True, exist_ok=True); _ensure_dirs(DET)

n_det = prepare_vnpollenbee(None, RAW, DET, local_dir=VPB_DIR)
n_cls = prepare_varroa(RAW, CLS, limit=VARROA_LIMIT)
write_data_yaml(DET)
print(f"\nDetection: {n_det} images (bee/pollen). Varroa-cls: {n_cls} crops.")

In [ ]:
# === 4. Sanity: detection class dist (0=bee 1=pollen) + varroa-cls counts ===
from collections import Counter
c = Counter()
for txt in Path("datasets/data").rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip(): c[int(line.split()[0])] += 1
print("detection instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path("datasets/data") / s / "images"
    print(f"  {s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")
print("varroa classification crops:")
for split in ["train", "val", "test"]:
    for lab in ["healthy", "varroa"]:
        p = Path("datasets/varroa_cls") / split / lab
        print(f"  {split}/{lab}: {len(list(p.glob('*'))) if p.is_dir() else 0}")

In [ ]:
# === 5. Zip BOTH + save to Google Drive ===
from google.colab import drive
drive.mount('/content/drive')
DET_ZIP = '/content/drive/MyDrive/bee_dataset.zip'   # detection (stage 1)
CLS_ZIP = '/content/drive/MyDrive/varroa_cls.zip'    # classification (stage 2)
shutil.make_archive(DET_ZIP[:-4], 'zip', root_dir='datasets', base_dir='data')
shutil.make_archive(CLS_ZIP[:-4], 'zip', root_dir='datasets', base_dir='varroa_cls')
print(f"Saved {DET_ZIP}  ({os.path.getsize(DET_ZIP)/1e6:.0f} MB)")
print(f"Saved {CLS_ZIP}  ({os.path.getsize(CLS_ZIP)/1e6:.0f} MB)")
print("\nNext: share-link each zip (right-click -> Share -> Anyone with link):")
print("  bee_dataset.zip  -> colab_train.ipynb DATASET_URL")
print("  varroa_cls.zip   -> colab_train_varroa_cls.ipynb DATASET_URL")